            # Surrogate v2 — Plan + Knobs + Workload → Latency / Ranking

            This notebook implements the pipeline you outlined (Phases 1–5):

            - Phase 1: DB-agnostic representations for query plans, knobs, and workloads
            - Phase 2: Train a PostgreSQL regression baseline (RMSE + Spearman)
            - Phase 3: Train a ranking model (LightGBM LambdaRank)
            - Phase 4: Transfer to MySQL (warm start + weighting + pseudo-labels)
            - Phase 5: Validate with Top-k + NDCG (per workload)

            Uses repo CSVs:
            - surrogate/cost_model_collected.csv
            - surrogate/cost_model_run_history.csv
            


In [ ]:
# If you don't have LightGBM installed in your env, uncomment:
# !pip install lightgbm joblib scikit-learn pandas numpy

from __future__ import annotations

import ast
import json
import math
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path.cwd()  # run from repo root (/home/E2ETune-AI4DB)
COLLECTED_CSV = ROOT / 'surrogate' / 'cost_model_collected.csv'
RUN_HISTORY_CSV = ROOT / 'surrogate' / 'cost_model_run_history.csv'
ARTIFACT_DIR = ROOT / 'surrogate' / 'artifacts' / 'transfer_rank_surrogate'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PLAN_REPR = os.environ.get('PLAN_REPR', 'embedding')  # 'embedding' | 'structured'
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

print('PLAN_REPR =', PLAN_REPR)
print('Collected CSV:', COLLECTED_CSV)
print('Run history CSV:', RUN_HISTORY_CSV)
print('Artifact dir:', ARTIFACT_DIR)


In [ ]:
def safe_parse_json_list(value: Any) -> Optional[list]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return None
    s = value.strip()
    if not s:
        return None
    try:
        parsed = json.loads(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        return None

def safe_parse_emb_vector(value: Any) -> Optional[np.ndarray]:
    lst = safe_parse_json_list(value)
    if lst is None:
        return None
    try:
        arr = np.asarray(lst, dtype=float)
        if arr.ndim != 1 or arr.size == 0:
            return None
        return arr
    except Exception:
        return None

collected = pd.read_csv(COLLECTED_CSV)
run_hist = pd.read_csv(RUN_HISTORY_CSV)

KEY_COL = 'metadata.workload_key'
TARGET_COL = 'target.cost'

print('collected:', collected.shape)
print('run_hist:', run_hist.shape)

# Merge knob configs with workload-level features/plans
common_keys = [
    c
    for c in [
        'metadata.benchmark',
        'metadata.db_engine',
        'metadata.hardware',
        'metadata.hardware_specs.cores',
        'metadata.hardware_specs.ram_gb',
        'metadata.hardware_specs.threads',
        'metadata.workload',
        KEY_COL,
    ]
    if c in run_hist.columns and c in collected.columns
]

keep_cols = [
    c
    for c in collected.columns
    if c.startswith('collected.') or c.startswith('metadata.') or c == 'qp_emb_vector'
]

merged = run_hist.merge(collected[keep_cols], on=common_keys, how='left')

if TARGET_COL not in merged.columns:
    raise KeyError(f'Missing {TARGET_COL} after merge')

y = merged[TARGET_COL].astype(float).to_numpy()
wk = merged[KEY_COL].astype(str).to_numpy()

print('merged:', merged.shape)
print('y stats:', float(np.nanmin(y)), float(np.nanmean(y)), float(np.nanmax(y)))


In [ ]:
# Phase 1.1 — Query plan encoding (DB-agnostic structured features)
# Your collected.query_plans strings already look like a portable operator-tree serialization.

OP_ALIASES = {
    # Scans
    'seq scan': 'table scan',
    'table scan': 'table scan',
    'index scan': 'index scan',
    'index only scan': 'index scan',
    # Joins
    'hash join': 'join',
    'merge join': 'join',
    'nested loop': 'join',
    'join': 'join',
    # Sort/Agg
    'sort': 'sort',
    'aggregate': 'aggregate',
    # Misc
    'limit': 'limit',
    'gather': 'gather',
    'gather merge': 'gather',
}

def normalize_op(op: str) -> str:
    s = (op or '').strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return OP_ALIASES.get(s, s)

def structured_plan_features(plan_list: List[str]) -> Dict[str, float]:
    # Best-effort: extract operator tokens + cost/rows + join/index signals.
    feats: Dict[str, float] = {}
    if not plan_list:
        return feats

    ops: List[str] = []
    costs: List[float] = []
    rows: List[float] = []
    any_index = 0
    join_hash = 0
    join_merge = 0
    join_nested = 0

    for p in plan_list:
        if not isinstance(p, str) or not p.strip():
            continue
        # operators are the tokens before '('
        for raw in re.findall(r'([A-Za-z][A-Za-z ]+?)\(', p):
            norm = normalize_op(raw)
            ops.append(norm)
            rl = raw.lower()
            join_hash += int('hash join' in rl)
            join_merge += int('merge join' in rl)
            join_nested += int('nested loop' in rl)
            any_index = any_index or int('index' in rl)

        for m in re.finditer(r'cost=([0-9]+(?:\.[0-9]+)?)', p):
            costs.append(float(m.group(1)))
        for m in re.finditer(r'rows=([0-9]+(?:\.[0-9]+)?)', p):
            rows.append(float(m.group(1)))

    if not ops:
        return feats

    total = float(len(ops))
    feats['plan.num_plans'] = float(len(plan_list))
    feats['plan.total_nodes'] = total
    feats['plan.any_index'] = float(any_index)
    feats['plan.join_hash_prop'] = float(join_hash) / total
    feats['plan.join_merge_prop'] = float(join_merge) / total
    feats['plan.join_nested_prop'] = float(join_nested) / total
    feats['plan.avg_cost'] = float(np.mean(costs)) if costs else 0.0
    feats['plan.max_cost'] = float(np.max(costs)) if costs else 0.0
    feats['plan.avg_rows'] = float(np.mean(rows)) if rows else 0.0
    feats['plan.max_rows'] = float(np.max(rows)) if rows else 0.0

    # Operator proportions
    counts: Dict[str, int] = {}
    for op in ops:
        counts[op] = counts.get(op, 0) + 1
    for op, cnt in counts.items():
        feats[f'plan.op_prop.{op}'] = float(cnt) / total

    return feats

# Compute structured plan features per workload_key and map to every config row
plan_struct = collected[[KEY_COL, 'collected.query_plans']].copy()
plan_struct['__plans'] = plan_struct['collected.query_plans'].apply(safe_parse_json_list)

feat_rows = []
for wk_i, plans in zip(plan_struct[KEY_COL].astype(str), plan_struct['__plans']):
    f = structured_plan_features(plans or [])
    f[KEY_COL] = wk_i
    feat_rows.append(f)

plan_struct_df = pd.DataFrame(feat_rows).fillna(0.0)
merged_struct = merged[[KEY_COL]].astype(str).merge(plan_struct_df, on=KEY_COL, how='left').fillna(0.0)

plan_struct_cols = [c for c in merged_struct.columns if c != KEY_COL]
plan_struct_matrix = merged_struct[plan_struct_cols].to_numpy(dtype=float)

print('structured plan dims:', plan_struct_matrix.shape)
print('example plan cols:', plan_struct_cols[:12])


In [ ]:
# Phase 1.2 — Knob encoding: normalize to [0, 1] with log1p for memory-like knobs
# Phase 1.3 — Workload encoding: fixed vector from collected.workload_features.*

workload_cols = [c for c in merged.columns if c.startswith('collected.workload_features.')]
knob_cols = [c for c in merged.columns if c.startswith('features.')]

LOG_HINTS = ('buffer', 'mem', 'cache', 'size', 'capacity', 'tmp', 'wal', 'shared', 'work_mem')
log_knob_cols = [c for c in knob_cols if any(h in c.lower() for h in LOG_HINTS)]

def encode_knobs(df: pd.DataFrame, cols: List[str], log_cols: List[str]) -> Tuple[np.ndarray, Dict[str, Dict[str, float]]]:
    Xk = df[cols].copy()
    for c in cols:
        Xk[c] = pd.to_numeric(Xk[c], errors='coerce')
    for c in log_cols:
        x = Xk[c].where(Xk[c] >= 0)
        Xk[c] = np.log1p(x)

    stats: Dict[str, Dict[str, float]] = {}
    for c in cols:
        arr = Xk[c].to_numpy(dtype=float)
        mn = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        mx = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else mn
        denom = (mx - mn) if (mx - mn) != 0 else 1.0
        Xk[c] = ((Xk[c] - mn) / denom).clip(0.0, 1.0)
        stats[c] = {'min': mn, 'max': mx}

    return Xk.fillna(0.0).to_numpy(dtype=float), stats

knob_matrix, knob_stats = encode_knobs(merged, knob_cols, log_knob_cols)
workload_matrix = merged[workload_cols].fillna(0.0).to_numpy(dtype=float)

print('knob dims:', knob_matrix.shape)
print('workload dims:', workload_matrix.shape)
print('log knobs:', len(log_knob_cols))


In [ ]:
# Phase 1.4 — Final input: X = [plan_features, knobs, workload]

def get_plan_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, object]]:
    if PLAN_REPR == 'structured':
        return plan_struct_matrix, {'plan_repr': 'structured', 'cols': plan_struct_cols}

    # PLAN_REPR == 'embedding' expects qp_emb_vector already filled
    if 'qp_emb_vector' not in df.columns:
        raise KeyError('Missing qp_emb_vector; run surrogate/add_query_plan_embeddings.py or set PLAN_REPR=structured')

    vecs = df['qp_emb_vector'].apply(safe_parse_emb_vector).tolist()
    dims = [v.size for v in vecs if v is not None]
    if not dims:
        raise ValueError('No valid qp_emb_vector found; set PLAN_REPR=structured or backfill embeddings')

    dim = int(np.median(dims))
    mat = np.zeros((len(vecs), dim), dtype=float)
    missing = 0
    for i, v in enumerate(vecs):
        if v is None:
            missing += 1
            continue
        if v.size != dim:
            raise ValueError(f'Embedding dim mismatch at row {i}: {v.size} != {dim}')
        mat[i, :] = v

    return mat, {'plan_repr': 'embedding', 'dim': dim, 'missing': missing}

plan_matrix, plan_meta = get_plan_matrix(merged)
X = np.hstack([plan_matrix, knob_matrix, workload_matrix]).astype(float)

print('plan dims:', plan_matrix.shape)
print('X dims:', X.shape)
print('plan_meta:', plan_meta)


In [ ]:
# Phase 2 — Regression baseline on PostgreSQL

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    s1 = pd.Series(y_true).rank(method='average')
    s2 = pd.Series(y_pred).rank(method='average')
    return float(s1.corr(s2))

def eval_per_workload(y_true: np.ndarray, y_pred: np.ndarray, wk_keys: np.ndarray) -> Dict[str, float]:
    df = pd.DataFrame({'wk': wk_keys, 'y': y_true, 'p': y_pred})
    spears = []
    top1 = []
    top3 = []
    for _, g in df.groupby('wk'):
        if len(g) < 2:
            continue
        spears.append(spearman_corr(g['y'].to_numpy(), g['p'].to_numpy()))
        # lower latency is better
        best_true = int(g['y'].idxmin())
        top1_pred = int(g['p'].idxmin())
        top3_pred = g.nsmallest(min(3, len(g)), 'p').index.tolist()
        top1.append(1.0 if top1_pred == best_true else 0.0)
        top3.append(1.0 if best_true in top3_pred else 0.0)
    return {
        'spearman_mean': float(np.nanmean(spears)) if spears else float('nan'),
        'top1_acc': float(np.mean(top1)) if top1 else float('nan'),
        'top3_recall': float(np.mean(top3)) if top3 else float('nan'),
    }

engine = (
    merged['metadata.db_engine'].astype(str).str.lower()
    if 'metadata.db_engine' in merged.columns
    else pd.Series(['postgresql'] * len(merged))
)
is_pg = (engine == 'postgresql').to_numpy()

X_pg = X[is_pg]
y_pg = y[is_pg]
wk_pg = wk[is_pg]

gkf = GroupKFold(n_splits=min(5, len(np.unique(wk_pg))))
train_idx, test_idx = next(iter(gkf.split(X_pg, y_pg, groups=wk_pg)))

X_train, X_test = X_pg[train_idx], X_pg[test_idx]
y_train, y_test = y_pg[train_idx], y_pg[test_idx]
wk_test = wk_pg[test_idx]

try:
    from lightgbm import LGBMRegressor
    reg = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_SEED,
    )
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    reg = HistGradientBoostingRegressor(random_state=RANDOM_SEED)

reg.fit(X_train, y_train)
pred = reg.predict(X_test)

rmse = math.sqrt(mean_squared_error(y_test, pred))
per_wk = eval_per_workload(y_test, pred, wk_test)

print('PG regression RMSE:', rmse)
print('PG per-workload:', per_wk)


In [ ]:
# Phase 3 + 4 — Ranking model + transfer to MySQL

def relevance_labels(lat: np.ndarray, wk_keys: np.ndarray, levels: int = 5) -> np.ndarray:
    rel = np.zeros_like(lat, dtype=int)
    df = pd.DataFrame({'wk': wk_keys, 'y': lat})
    for _, idx in df.groupby('wk').groups.items():
        y_g = df.loc[idx, 'y']
        ranks = (y_g.rank(method='first', ascending=True).to_numpy() - 1)
        if len(ranks) == 1:
            rel[idx] = levels - 1
            continue
        q = ranks / max(1, (len(ranks) - 1))
        rel[idx] = np.floor((1.0 - q) * (levels - 1) + 1e-9).astype(int)
    return rel

def group_sizes(sorted_wk: np.ndarray) -> List[int]:
    sizes = []
    last = None
    cnt = 0
    for k in sorted_wk:
        if last is None:
            last = k
            cnt = 1
        elif k == last:
            cnt += 1
        else:
            sizes.append(cnt)
            last = k
            cnt = 1
    if last is not None:
        sizes.append(cnt)
    return sizes

try:
    from lightgbm import LGBMRanker
except Exception as e:
    raise RuntimeError('LightGBM is required for LambdaRank. Install: pip install lightgbm') from e

order_pg = np.argsort(wk_pg)
X_pg_s = X_pg[order_pg]
y_pg_s = y_pg[order_pg]
wk_pg_s = wk_pg[order_pg]

rel_pg = relevance_labels(y_pg_s, wk_pg_s, levels=5)
groups_pg = group_sizes(wk_pg_s)

ranker = LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_SEED,
)
ranker.fit(X_pg_s, rel_pg, group=groups_pg)

# Transfer to MySQL (if present)
is_my = (engine == 'mysql').to_numpy()
if not np.any(is_my):
    print('No MySQL rows found; skipping transfer.')
else:
    # Optional domain feature
    db_type = (engine == 'mysql').astype(int).to_numpy().reshape(-1, 1)
    X_domain = np.hstack([X, db_type])

    X_my = X_domain[is_my]
    y_my = y[is_my]
    wk_my = wk[is_my]

    # Pseudo labels (weight 0.3) + real labels (weight 1.0)
    pseudo_scores = ranker.predict(X_my)
    pseudo_rel = relevance_labels(-pseudo_scores, wk_my, levels=5)
    real_rel = relevance_labels(y_my, wk_my, levels=5)

    X_ft = np.vstack([X_my, X_my])
    rel_ft = np.concatenate([real_rel, pseudo_rel])
    wk_ft = np.concatenate([wk_my, wk_my])
    w_ft = np.concatenate([
        np.ones_like(real_rel, float),
        np.ones_like(pseudo_rel, float) * 0.3,
    ])

    order = np.argsort(wk_ft)
    X_ft_s = X_ft[order]
    rel_ft_s = rel_ft[order]
    wk_ft_s = wk_ft[order]
    w_ft_s = w_ft[order]
    groups_ft = group_sizes(wk_ft_s)

    ranker_my = LGBMRanker(
        objective='lambdarank',
        metric='ndcg',
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_SEED,
    )

    ranker_my.fit(
        X_ft_s,
        rel_ft_s,
        group=groups_ft,
        sample_weight=w_ft_s,
        init_model=ranker.booster_,
    )

    score_my = ranker_my.predict(X_my)
    per_wk_my = eval_per_workload(y_my, -score_my, wk_my)
    print('MySQL per-workload (using -score as predicted latency):', per_wk_my)


In [ ]:
# Phase 5 — Save artifacts

import joblib

meta = {
    'plan_meta': plan_meta,
    'plan_repr': PLAN_REPR,
    'knob_cols': knob_cols,
    'log_knob_cols': log_knob_cols,
    'knob_minmax': knob_stats,
    'workload_cols': workload_cols,
    'target_col': TARGET_COL,
    'key_col': KEY_COL,
}

joblib.dump(meta, ARTIFACT_DIR / 'feature_meta.joblib')
joblib.dump(reg, ARTIFACT_DIR / 'pg_regressor.joblib')
joblib.dump(ranker, ARTIFACT_DIR / 'pg_ranker.joblib')
if 'ranker_my' in globals():
    joblib.dump(ranker_my, ARTIFACT_DIR / 'mysql_ranker_finetuned.joblib')

print('Saved artifacts to:', ARTIFACT_DIR)
